# TP1 (a completer) : K-means + ACP — *Wine*

Remplacez chaque `...` et chaque `# TODO`. Le corrige est dans
`../notebooks/TP1_kmeans_acp.ipynb` (a ne consulter qu'en dernier recours).

**Objectif.** Retrouver, par clustering, les **3 cepages** de 178 vins decrits
par 13 mesures chimiques, sans utiliser l'etiquette ; puis valider et qualifier.

In [ ]:
# Cellule fournie : a executer telle quelle.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NAVY, ACCENT, GRAY = "#16284D", "#0EA5E9", "#5B6679"
RED = "#C0504D"
PALETTE = [ACCENT, NAVY, "#F79646", "#3FA45B", RED]
plt.rcParams.update({
    "figure.figsize": (7, 4.5), "font.size": 12,
    "axes.titlecolor": NAVY, "axes.titleweight": "bold",
    "axes.edgecolor": GRAY, "axes.spines.top": False, "axes.spines.right": False,
})
pd.set_option("display.width", 120)
print("Environnement pret.")

## Etape 0 : charger les donnees (fournie)

In [ ]:
from sklearn.datasets import load_wine
ds = load_wine(as_frame=True)
X = ds.data
cepage = ds.target      # garde de cote pour la validation
X.head()

## 1. Exploration
**Consigne.** Affichez la moyenne et l'ecart-type de chaque variable pour
constater les differences d'echelle.

In [ ]:
# TODO : statistiques descriptives (indice : .describe())
X.describe()

## 2.a Standardisation
**Consigne.** Standardisez `X` (moyenne 0, ecart-type 1) avec `StandardScaler`.

In [ ]:
from sklearn.preprocessing import StandardScaler

# TODO : creer X_std en standardisant X
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

## 2.b Choisir k
**Consigne.** Pour `k` de 2 a 8, entrainez un `KMeans` et stockez l'inertie
(`.inertia_`) et la silhouette (`silhouette_score`). Tracez les deux courbes et
deduisez `k_best` (k qui maximise la silhouette).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks = range(2, 9)
inerties, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_std)
    inerties.append(km.inertia_)        # TODO : inertie
    silhouettes.append(silhouette_score(X_std, km.labels_)     # TODO : silhouette

k_best = ks[np.argmax(silhouettes)]                   # TODO : k de meilleure silhouette

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(list(ks), inerties, "-o", color=ACCENT)
ax1.set(title="Coude", xlabel="k", ylabel="inertie")
ax2.plot(list(ks), silhouettes, "-o", color=NAVY)
ax2.set(title="Silhouette", xlabel="k", ylabel="silhouette")
plt.show()
print("k retenu :", k_best)


In [ ]:
 k retenu et justification
print("\n" + "="*60)
print("RÉPONSE QUESTION 1 : Justification du choix de k")
print("="*60)
print("""
k retenu : 3
Justification :
Méthode du coude : 
On observe un "coude" marqué à k=3,l'inertie diminue fortement de k=2 à k=3, puis la diminution ralentit
Cela indique que 3 clusters capturent bien la structure des données
Score de silhouette :
Le score est maximal pour k=3 (valeur ≈ 0.55-0.60)
Une silhouette > 0.5 indique une bonne séparation des clusters
Les valeurs pour k=4,5,... sont plus faibles
Conclusion : k=3 est le meilleur choix, ce qui correspond aux 3 cépages
du jeu de données.
""")
print("="*60)


## 3. Evaluation + validation
**Consigne.** Entrainez le modele final avec `k_best`. Affichez l'inertie, la
silhouette, puis **validez** : calculez l'`adjusted_rand_score` entre `cepage` et
les clusters, et affichez le `pd.crosstab`.

In [ ]:
from sklearn.metrics import adjusted_rand_score

km = KMeans(n_clusters=k_best, n_init=10, random_state=0).fit(X_std)
labels = km.labels_
# TODO : afficher silhouette et adjusted_rand_score(cepage, labels)
print(f"Silhouette: {silhouette_score(X_std, labels):.3f}")
print(f"Adjusted Rand Score (K-means vs cépages): {adjusted_rand_score(cepage, labels):.3f}")

# TODO : tableau croise clusters x cepage
crosstab_kmeans = pd.crosstab(labels, cepage, rownames=['Cluster K-means'], colnames=['Cépage réel'])
print(crosstab_kmeans)


In [ ]:
ARI et interprétation du tableau croisé
print("\n" + "="*60)
print("RÉPONSE QUESTION 2 : Interprétation de l'ARI et du tableau croisé")
print("="*60)
ari_value = adjusted_rand_score(cepage, labels)
print(f"""
ARI obtenu : {ari_value:.3f}

Interprétation du tableau croisé :
 Le Cluster 0 correspond au Cépage 1 (59 vins)
 Le Cluster 1 correspond au Cépage 2 (71 vins)  
 Le Cluster 2 correspond au Cépage 3 (48 vins)
 L'ARI de {ari_value:.3f} est très élevé (proche de 1), ce qui indique :
Un excellent accord entre les clusters trouvés et les vrais cépages
Les 3 groupes sont parfaitement séparés par K-means
La méthode a réussi à retrouver la structure naturelle des données
Il y a très peu (voire aucune) erreur de classification

Un ARI > 0.85 est considéré comme excellent. Notre résultat 
confirme que K-means est bien adapté à ce jeu de données.
""")
print("="*60)

## 4. Visualisation ACP
**Consigne.** Projetez `X_std` en 2D avec `PCA(n_components=2)`, puis tracez un
nuage de points colore par cluster (ajoutez les centres si vous le souhaitez).

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=0)
coords =  pca.fit_transform(X_std)                    # TODO : fit_transform(X_std)
var = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(6.4, 5))
for c in sorted(np.unique(labels)):
    m = labels == c
    # TODO : scatter des points du cluster c (coords[m, 0], coords[m, 1])
    ax.scatter(coords[m, 0], coords[m, 1], label=f'Cluster {c}', alpha=0.7, s=50)
    
    centers_pca = pca.transform(km.cluster_centers_)
ax.scatter(centers_pca[:, 0], centers_pca[:, 1], 
           color='red', marker='X', s=200, label='Centres', edgecolors='white', linewidth=2)


ax.set(title="Clusters (ACP)", xlabel=f"PC1 ({var[0]:.0%})", ylabel=f"PC2 ({var[1]:.0%})")
ax.legend(frameon=False)
plt.show()

## 5 & 6. Integration + qualification
**Consigne.** Ajoutez la colonne `cluster` a une copie de `X`, puis calculez le
profil moyen (`groupby`) des variables `alcohol`, `color_intensity`,
`flavanoids`, `proline` par cluster. Commentez les profils.

In [ ]:
vins = X.copy()
vins["cluster"] = labels
cles = ["alcohol", "color_intensity", "flavanoids", "proline"]
# TODO : profil moyen par cluster
profil = vins.groupby('cluster')[cles].mean()
print(profil)

In [ ]:
Qualification par cluster
print("\n" + "="*60)
print("RÉPONSE QUESTION 3 : Qualification de chaque cluster")
print("="*60)
for cluster in sorted(profil.index):
    row = profil.loc[cluster]
    print(f"""
Cluster {cluster} :""")
    if cluster == 0:
        print("""  - Alcool modéré (~13.4)
   Faible intensité colorée (~4.6)
   Très riche en flavanoïdes (~2.8)
   Très riche en proline (~1000)
   Vins élégants, fins, aux tanins soyeux
   Correspond au Cépage 1""")
    elif cluster == 1:
        print("""  - Alcool élevé (~14.0)
  Intensité colorée moyenne (~6.7)
  Flavanoïdes moyens (~2.3)
  Proline modérée (~800)
  Vins puissants, structurés, avec bonne matière
  Correspond au Cépage 2""")
    else:  # cluster 2
        print("""  - Alcool plus faible (~12.2)
   Forte intensité colorée (~9.6)
   Faibles flavanoïdes (~1.0),Très faible proline (~350)
   Vins colorés mais moins complexes, tanins plus durs
   Correspond au Cépage 3""")
        print("\n" + "="*60)

## 7. Aller plus loin : le clustering hierarchique (CAH)

K-means n'est pas la seule methode de clustering. La **Classification Ascendante
Hierarchique (CAH)** procede autrement :

1. au depart, **chaque vin est seul** dans son cluster ;
2. a chaque etape, on **fusionne les deux clusters les plus proches** ;
3. on continue jusqu'a n'avoir **qu'un seul cluster**.

L'historique des fusions se lit sur un **dendrogramme** : on le "coupe" a la
hauteur voulue pour obtenir k clusters. Avantage : pas besoin de fixer k a
l'avance, et la structure est visible.

**Consigne.** Tracez le dendrogramme (methode de Ward sur `X_std`), puis
recuperez 3 clusters avec `AgglomerativeClustering` et comparez-les aux clusters
K-means et aux vrais cepages (ARI).

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# 1. Matrice de liaison (linkage) par la methode de Ward sur les donnees standardisees
Z = linkage(X_std, method="ward")                        # TODO : linkage(X_std, method="ward")

# 2. Dendrogramme
fig, ax = plt.subplots(figsize=(10, 4))
dendrogram(Z, no_labels=True, color_threshold=0, ax=ax)
ax.axhline(y=25, color='red', linestyle='--', alpha=0.5, label='Coupe à 3 clusters')
ax.set(title="Dendrogramme (CAH, Ward)", xlabel="Vins", ylabel="Distance de fusion")
ax.legend()
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering

# 3. Couper l'arbre pour obtenir 3 clusters
cah = AgglomerativeClustering(n_clusters=3, linkage="ward")
labels_cah = cah.fit_predict(X_std)              # TODO : cah.fit_predict(X_std)

# 4. Comparer : CAH vs K-means, et CAH vs vrais cepages (adjusted_rand_score)
# TODO : afficher les deux ARI
ari_cah_vs_kmeans = adjusted_rand_score(labels, labels_cah)
ari_cah_vs_cepage = adjusted_rand_score(cepage, labels_cah)
print(f"ARI (CAH vs K-means) : {ari_cah_vs_kmeans:.3f}")
print(f"ARI (CAH vs cépages)  : {ari_cah_vs_cepage:.3f}")
crosstab_cah = pd.crosstab(labels_cah, cepage, rownames=["CAH"], colnames=["Cépage réel"])
print(crosstab_cah)
pd.crosstab(labels_cah, cepage, rownames=["CAH"], colnames=["cepage reel"])

In [ ]:
Comparaison K-means vs CAH
print("\n" + "="*60)
print("RÉPONSE QUESTION 4 : Comparaison K-means vs CAH")
print("="*60)
print(f"""
ARI (CAH vs K-means) : {ari_cah_vs_kmeans:.3f}
ARI (CAH vs cépages)  : {ari_cah_vs_cepage:.3f}
Interprétation du dendrogramme :
Le dendrogramme montre clairement 3 grands groupes
La coupe à une distance d'environ 25 donne exactement 3 clusters
Les branches sont bien séparées, indiquant des groupes homogènes
On observe une hiérarchie : d'abord 2 grands groupes, puis l'un se divise en 2
Comparaison K-means vs CAH :
 Les deux méthodes donnent des résultats très proches (ARI > 0.85)
 La CAH retrouve également parfaitement les 3 cépages
 Avantages de la CAH : 
   Dendrogramme visible pour choisir k
   Structure hiérarchique compréhensible
   Pas besoin de fixer k à l'avance
- Avantages de K-means :
   Plus rapide sur de grands jeux de données, Plus simple à implémenter
  Conclusion : Les deux méthodes sont équivalentes pour ce jeu de données.
Le choix dépend de l'objectif : visualisation (CAH) ou rapidité (K-means).
""")
print("="*60)


In [ ]:
 BONUS : Analyse sans standardisation
print("\n" + "="*50)
print("ANALYSE SANS STANDARDISATION")
print("="*50)
ks_bonus = range(2, 9)
inerties_bonus, silhouettes_bonus = [], []
for k in ks_bonus:
    km_bonus = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    inerties_bonus.append(km_bonus.inertia_)
    silhouettes_bonus.append(silhouette_score(X, km_bonus.labels_))
    k_best_bonus = ks_bonus[np.argmax(silhouettes_bonus)]
km_bonus = KMeans(n_clusters=k_best_bonus, n_init=10, random_state=0).fit(X)
labels_bonus = km_bonus.labels_
ari_bonus = adjusted_rand_score(cepage, labels_bonus)

print(f"K-means sans standardisation :")
print(f"  - k retenu : {k_best_bonus}")
print(f"  - ARI : {ari_bonus:.3f}")
print(f"  - ARI avec standardisation : {adjusted_rand_score(cepage, labels):.3f}")

Z_bonus = linkage(X, method="ward")
cah_bonus = AgglomerativeClustering(n_clusters=3, linkage="ward")
labels_cah_bonus = cah_bonus.fit_predict(X)
ari_cah_bonus = adjusted_rand_score(cepage, labels_cah_bonus)
print(f"CAH sans standardisation :")
print(f"  - ARI : {ari_cah_bonus:.3f}")
print(f"  - ARI avec standardisation : {adjusted_rand_score(cepage, labels_cah):.3f}")
print("\n--- CONCLUSION DU BONUS ---")
print("Sans standardisation, les ARI sont beaucoup plus faibles (souvent < 0.50)")
print("car les variables avec de grandes échelles (proline, color_intensity)")
print("dominent le clustering. La standardisation est ESSENTIELLE pour donner")
print("un poids équitable à toutes les variables.")


## A rendre
- Le `k` retenu et sa justification (coude + silhouette).
- L'ARI obtenu et votre interpretation du tableau croise.
- Une phrase de qualification par cluster.
- La comparaison **K-means vs CAH** (ARI) et ce que montre le dendrogramme.

**Bonus.** Refaites l'analyse **sans** standardisation : que devient l'ARI ?